# Who: Jacopo

**Task:** Make text-classifier on the consult texts with agenda codes as labels

*Ingredients*: 
* probably best to use a trusted CNN but I know that you want to use a transformer so..just follow the recipe you can probably find online ;)
* Your text column:  ```text```, contains a concatenation of ```SpecialismeNaam, title, section_title_display_original, section_text```
* The target variable: use the target value you extracted from the afspraken set
* for the training use cross-validation: we are interested firstly in the performance on the test-sets, AND we are interested in **inspecting** the samples in the test-set that are high confidence false negative (potentially patients that should be included) and high confidence false negatives. 


In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df_consult = pd.read_parquet(r"T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240826\consult_texts.parquet")

In [3]:
df_afsprak = pd.read_sas(
    r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240822\consult_20240822.sas7bdat',
        encoding='latin1')
df_afsprak.studyId_0831 = df_afsprak.studyId_0831.astype('uint64')

In [4]:
df_afsprak = df_afsprak.drop(['Consult_id', 'ConsultStatus', 'type_code_original',
                              'type_display_original','created', 'title', 'author_usercode',
                              'initial_author_usercode', 'custodian_Organization_value',
                              'SpecialismeNaam', 'initial_custodian_Organization_v',
                              'SpecialismeNaamInitial', 'LocationCode', 'LocationName',
                              'PatientProtected', 'section_title_code_original',
                              'section_title_display_original', 'section_text',
                              'section_text_contentType', 'orderedBy', 'index_date'],axis='columns')

In [ ]:
df_afsprak = df_afsprak.groupby('studyId_0831').last()
df_afsprak

In [ ]:
df_merged = pd.merge(df_consult, df_afsprak, right_index=True,left_on='studyId_0831',how='left')
df_merged

In [ ]:
df_merged.query('studyId_0831 == 7992019')

In [8]:
df_merged['delta'] = (df_merged.date_x - df_merged.date_y).dt.days
df_merged_filt = df_merged.query('delta >= -365 and delta <= 0')


In [ ]:
df_merged_filt.query('studyId_0831 == 7992019')

In [10]:
indices = []
def take_longer_consult(x):
    all_consults = x.to_list()
    max_len = -1
    for i,c in enumerate(all_consults):
        if len(c) > max_len:
            tmp = c
            idx = i
            max_len = len(c)
    
    indices.append(x.index[idx])
    return tmp


In [ ]:
df_merged_f_g = df_merged_filt[['studyId_0831','text']]\
                    .groupby('studyId_0831')\
                        .agg(take_longer_consult)
df_merged_f_g

In [ ]:
df_merged_filt['delta'].loc[indices].hist(bins=100)


In [13]:
df_merged_f_g['delta'] = df_merged_filt['delta'].loc[indices]

In [14]:
tokenizer = AutoTokenizer.from_pretrained("CLTL/MedRoBERTa.nl")
model = AutoModelForMaskedLM.from_pretrained("CLTL/MedRoBERTa.nl")

In [15]:
def splitter_512(t):
    sections = []
    how_many_512 = len(t)//512
    if how_many_512 > 1:
        for i in range(how_many_512):
            sections.append(t[i*512: (512*i) + 512])

        sections.append(t[(512*i) + 512:])
    else:
        sections.append(t)
    return sections

In [16]:
df_cooked = pd.DataFrame(columns=['ids'])
#df_cooked['chunked_text'] = df_merged_f_g.text.apply(splitter_512)
#df_cooked

In [17]:
def to_ids(chunk):
    return tokenizer(chunk,return_attention_mask=False,return_tensors='pt',padding=True)['input_ids']

In [18]:
df_cooked['ids'] = df_merged_f_g.text.apply(to_ids)


In [19]:
df_cooked['id_len'] = df_cooked['ids'].apply(lambda x: x.shape[1])

In [ ]:
m = 0.01
M = 1-m
plt.hist(df_cooked['id_len'],bins=100)
plt.axvline(df_cooked['id_len'].quantile(m),color='red')
plt.axvline(df_cooked['id_len'].quantile(M),color='red')

In [ ]:
q_min = df_cooked['id_len'].quantile(m)
q_max = df_cooked['id_len'].quantile(M)
df_cooked.query('id_len > @q_min and id_len < @q_max')

In [ ]:
token_vector = model.roberta.embeddings(df_cooked['ids'].iloc[0][:,:511])
tokenv_cls = torch.concat([torch.randn((1,1,768)),token_vector],dim=1)
tokenv_cls.shape

In [ ]:
model.roberta.encoder(tokenv_cls).last_hidden_state[:,0,:].shape

In [ ]:
from jacopy.model import Consultformer
inputs = torch.concat([df_cooked['ids'].iloc[0][:,:511],df_cooked['ids'].iloc[0][:,:511]],dim=0)
net = Consultformer(2,2)

print(net(inputs).shape)